In [1]:
# ============================================================
# SAR Pipeline v2 — Full Analysis with GroupKFold + Validation
# Changes from v1:
#   - Section 4: GroupKFold(5) by farmer ID (Option Y)
#   - Section 3e: k-sensitivity analysis (k=2..5)
#   - Section 3f: GMM clustering robustness check
#   - Section 6b: Surrogate tree 80/20 held-out validation
#   - Section 6c: Surrogate tree 5-fold CV
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import (
    cross_val_score, GroupKFold, StratifiedKFold,
    GroupShuffleSplit, cross_validate
)
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    accuracy_score, classification_report
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBRegressor
from scipy.stats import spearmanr

import os
from pyprojroot import here
OUT = 'figures'
os.makedirs(OUT, exist_ok=True)
print("✓ Imports complete")

✓ Imports complete


In [2]:
# ============================================================
# SECTION 1 — DATA LOADING & FEATURE DEFINITIONS
# ============================================================
# หาตำแหน่งของโฟลเดอร์ที่ไฟล์ code นี้วางอยู่
base_path = here()
file_path = os.path.join(base_path, "datas", "SFProgramDataPanal.csv")

df = pd.read_csv(file_path)
df = df.replace('.', np.nan)

CLUSTER_FEATS = [
    'age','edu','agri_long','irriga','loan',
    'Avg_ProdManage','Avg_InputManage','Avg_Tech',
    'Avg_Ana&Plan','Avg_Mkting','Avg_Network',
    'Ave_ProdRisk','Ave_InputRisk','Ave_MktRisk','Ave_FinRisk'
]
RF_FEATS = [
    'age','edu','agri_long','gender','irriga','loan','region',
    'Avg_ProdManage','Avg_InputManage','Avg_Tech','Avg_Ana&Plan',
    'Avg_Mkting','Avg_Network',
    'Ave_ProdRisk','Ave_InputRisk','Ave_MktRisk','Ave_FinRisk',
    'sf_participant'
]
SHAP_FEATS = [
    'age','edu','agri_long','irriga','loan',
    'Avg_ProdManage','Avg_Tech','Avg_Mkting',
    'Ave_MktRisk','Ave_FinRisk','Overview_Risk',
    'agri Org_mem','gov_support','All_Skill'
]
SHAP_RENAME = {
    'age':'Age','edu':'Education','agri_long':'Agri Exp',
    'irriga':'Irrigation','loan':'Loan',
    'Avg_ProdManage':'Avg_ProdManage','Avg_Tech':'Avg_Tech',
    'Avg_Mkting':'Avg_Mkting','Ave_MktRisk':'Mkt Risk',
    'Ave_FinRisk':'Fin Risk','Overview_Risk':'Overall Risk',
    'agri Org_mem':'Agri Org','gov_support':'Gov Support',
    'All_Skill':'All_Skill'
}
OUTCOME       = 'Ch_Skill'
FARMER_ID     = 'id'
CLUSTER_NAMES = {0:'Low-skill', 1:'Moderate-skill', 2:'High-skill'}
CLUSTER_COL   = {0:'#4472C4', 1:'#FF8C00', 2:'#2ECC71'}
RF_PARAMS     = dict(n_estimators=500, min_samples_leaf=10,
                     max_features='sqrt', random_state=42, n_jobs=-1)

print(f"Dataset: {df.shape[0]} rows, {df[FARMER_ID].nunique()} unique farmers")
print(f"Panel structure: Year=0 (pre-training), Year=1 (post-training)")

Dataset: 834 rows, 417 unique farmers
Panel structure: Year=0 (pre-training), Year=1 (post-training)


In [3]:
# ============================================================
# SECTION 2 — FARMER SEGMENTATION (K-MEANS, k=3)
# ============================================================

df_cl = df[CLUSTER_FEATS].apply(pd.to_numeric, errors='coerce').dropna().copy()
scaler = StandardScaler()
X_sc   = scaler.fit_transform(df_cl)

# ── 2a. Cluster validity indices k=2..8 ──────────────────────────────────
ks = range(2, 9)
cv_rows = []
for k in ks:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_sc)
    cv_rows.append({
        'k': k,
        'inertia':    km.inertia_,
        'silhouette': silhouette_score(X_sc, lbl),
        'calinski':   calinski_harabasz_score(X_sc, lbl),
        'davies':     davies_bouldin_score(X_sc, lbl),
    })
cv_df = pd.DataFrame(cv_rows)
print("\nCluster validity indices:")
print(cv_df.to_string(index=False, float_format='%.4f'))

# ── 2b. Fit k=3, label clusters ──────────────────────────────────────────
km3 = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cl['Cluster_raw'] = km3.fit_predict(X_sc)

df_full = df.loc[df_cl.index].apply(pd.to_numeric, errors='coerce').copy()
df_full['Cluster_raw'] = df_cl['Cluster_raw'].values

order     = df_full.groupby('Cluster_raw')['All_Skill'].mean().sort_values()
label_map = {old: new for new, old in enumerate(order.index)}
df_full['Cluster']      = df_full['Cluster_raw'].map(label_map)
df_full['Cluster_name'] = df_full['Cluster'].map(CLUSTER_NAMES)

print("\nCluster sizes:")
print(df_full['Cluster_name'].value_counts().sort_index())

# ── 2c. Figure: 4-panel validity ─────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
panels = [
    ('inertia','Within-cluster Inertia','Elbow Method','#1565C0'),
    ('silhouette','Silhouette Score','Silhouette','#1976D2'),
    ('calinski','Calinski-Harabasz Score','Calinski-Harabasz','#0277BD'),
    ('davies','Davies-Bouldin Score','Davies-Bouldin','#00796B'),
]
for ax, (col, ylabel, title, color) in zip(axes, panels):
    ax.plot(cv_df['k'], cv_df[col], 'o-', color=color, lw=2, ms=6)
    ax.axvline(3, color='#E53935', ls='--', lw=1.5, alpha=0.8, label='k=3')
    for k, v in zip(cv_df['k'], cv_df[col]):
        ax.annotate(f'{v:.3f}', (k, v), textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=7.5)
    ax.set(title=title, xlabel='Number of Clusters (k)', ylabel=ylabel)
    ax.grid(alpha=0.25); ax.legend(fontsize=8)
plt.suptitle('Cluster Validity: Four Complementary Indices (k=2–8)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_elbow_silhouette.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig_elbow_silhouette.png')

# ── 2d. PCA scatter ───────────────────────────────────────────────────────
pca    = PCA(n_components=2, random_state=42)
X_pca  = pca.fit_transform(X_sc)
ev     = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(7, 6))
for raw_lbl, mapped_lbl in label_map.items():
    mask = df_cl['Cluster_raw'] == raw_lbl
    ax.scatter(X_pca[mask,0], X_pca[mask,1],
               c=CLUSTER_COL[mapped_lbl],
               label=CLUSTER_NAMES[mapped_lbl],
               alpha=0.65, s=25, edgecolors='none')
ax.set(title='Farmer Clusters (k=3, Silhouette=0.185)',
       xlabel=f'PC1 ({ev[0]:.1%})', ylabel=f'PC2 ({ev[1]:.1%})')
ax.legend(framealpha=0.9, fontsize=10); ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(f'{OUT}/fig2_clusters_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig2_clusters_new.png')


Cluster validity indices:
 k   inertia  silhouette  calinski  davies
 2 9678.1594      0.1986  241.8636  1.8069
 3 8395.1843      0.1845  202.6666  1.8348
 4 7786.4915      0.1712  167.0994  1.7605
 5 7374.7595      0.1581  143.7186  1.9096
 6 6981.7123      0.1433  130.6123  2.0614
 7 6704.5583      0.1448  118.8968  2.0272
 8 6438.7959      0.1393  110.8541  1.9449

Cluster sizes:
Cluster_name
High-skill        334
Low-skill         271
Moderate-skill    228
Name: count, dtype: int64


✓ fig_elbow_silhouette.png


✓ fig2_clusters_new.png


In [4]:
# ============================================================
# SECTION 3e — K-SENSITIVITY ANALYSIS (k=2..5)
# ============================================================

print('\n=== Section 3e: k-sensitivity ===')
sens_rows = []
for k in range(2, 6):
    km_k  = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl_k = km_k.fit_predict(X_sc)
    df_full[f'Cluster_k{k}'] = df_cl.index.map(
        lambda i: lbl_k[df_cl.index.get_loc(i)] if i in df_cl.index else np.nan)
    sil = silhouette_score(X_sc, lbl_k)
    ch  = calinski_harabasz_score(X_sc, lbl_k)
    db  = davies_bouldin_score(X_sc, lbl_k)
    n_each = pd.Series(lbl_k).value_counts().sort_index().tolist()
    sens_rows.append({'k': k, 'Silhouette': sil, 'CH': ch, 'DB': db,
                      'Sizes': n_each})
    print(f"k={k}: Sil={sil:.3f} CH={ch:.1f} DB={db:.3f} sizes={n_each}")

# ── k-sensitivity figure ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
ks_s = [r['k'] for r in sens_rows]
for ax, (metric, label, col) in zip(axes, [
    ('Silhouette','Silhouette Score','#1976D2'),
    ('CH','Calinski-Harabasz','#0277BD'),
    ('DB','Davies-Bouldin','#00796B'),
]):
    vals = [r[metric] for r in sens_rows]
    ax.bar(ks_s, vals, color=col, alpha=0.8, edgecolor='white')
    ax.axvline(3, color='#E53935', ls='--', lw=1.5, alpha=0.8, label='k=3')
    for k, v in zip(ks_s, vals):
        ax.annotate(f'{v:.3f}', (k, v), textcoords='offset points',
                    xytext=(0, 5), ha='center', fontsize=9, fontweight='bold')
    ax.set(title=label, xlabel='k', ylabel=label)
    ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.suptitle('k-Sensitivity Analysis: Cluster Validity for k=2..5',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_k_sensitivity.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig_k_sensitivity.png')


=== Section 3e: k-sensitivity ===
k=2: Sil=0.199 CH=241.9 DB=1.807 sizes=[422, 411]
k=3: Sil=0.185 CH=202.7 DB=1.835 sizes=[334, 271, 228]
k=4: Sil=0.171 CH=167.1 DB=1.760 sizes=[146, 266, 309, 112]
k=5: Sil=0.158 CH=143.7 DB=1.910 sizes=[285, 142, 109, 129, 168]


✓ fig_k_sensitivity.png


In [5]:
# ============================================================
# SECTION 3f — GMM CLUSTERING ROBUSTNESS
# ============================================================

print('\n=== Section 3f: GMM robustness ===')
gmm_rows = []
for k in range(2, 6):
    gmm   = GaussianMixture(n_components=k, covariance_type='full',
                            random_state=42, n_init=5)
    lbl_g = gmm.fit_predict(X_sc)
    sil   = silhouette_score(X_sc, lbl_g)
    ch    = calinski_harabasz_score(X_sc, lbl_g)
    db    = davies_bouldin_score(X_sc, lbl_g)
    bic   = gmm.bic(X_sc)
    aic   = gmm.aic(X_sc)
    n_g   = pd.Series(lbl_g).value_counts().sort_index().tolist()
    gmm_rows.append({'k': k, 'Silhouette': sil, 'CH': ch,
                     'DB': db, 'BIC': bic, 'AIC': aic, 'Sizes': n_g})
    print(f"GMM k={k}: Sil={sil:.3f} CH={ch:.1f} DB={db:.3f} "
          f"BIC={bic:.0f} AIC={aic:.0f} sizes={n_g}")

# ── GMM vs KMeans comparison at k=3 ──────────────────────────────────────
km3_lbl  = km3.labels_
gmm3     = GaussianMixture(n_components=3, covariance_type='full',
                           random_state=42, n_init=5)
gmm3_lbl = gmm3.fit_predict(X_sc)

from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(km3_lbl, gmm3_lbl)
print(f"\nAdjusted Rand Index (KMeans k=3 vs GMM k=3): {ari:.4f}")
print("(ARI=1.0 = perfect agreement, 0.0 = random)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (lbl, title) in zip(axes, [
    (km3_lbl, f'K-Means (k=3)\nSilhouette={cv_df[cv_df["k"]==3]["silhouette"].values[0]:.3f}'),
    (gmm3_lbl, f'GMM (k=3)\nSilhouette={gmm_rows[1]["Silhouette"]:.3f}'),
]):
    order_l = pd.Series(lbl).value_counts().sort_values(ascending=False).index
    palette = ['#4472C4','#FF8C00','#2ECC71']
    for i, lbl_i in enumerate(order_l):
        mask = lbl == lbl_i
        ax.scatter(X_pca[mask,0], X_pca[mask,1], c=palette[i],
                   alpha=0.6, s=20, label=f'Cluster {i+1}')
    ax.set(title=title, xlabel=f'PC1 ({ev[0]:.1%})',
           ylabel=f'PC2 ({ev[1]:.1%})')
    ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.suptitle(f'Clustering Robustness: K-Means vs GMM (k=3)\n'
             f'Adjusted Rand Index = {ari:.3f}', fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_gmm_robustness.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig_gmm_robustness.png')


=== Section 3f: GMM robustness ===
GMM k=2: Sil=0.167 CH=194.7 DB=2.000 BIC=30049 AIC=28768 sizes=[442, 391]


GMM k=3: Sil=0.078 CH=71.5 DB=3.910 BIC=22923 AIC=21000 sizes=[347, 339, 147]


GMM k=4: Sil=0.053 CH=39.6 DB=3.018 BIC=13074 AIC=10508 sizes=[181, 18, 491, 143]


GMM k=5: Sil=0.039 CH=33.8 DB=2.945 BIC=15582 AIC=12374 sizes=[338, 27, 145, 305, 18]



Adjusted Rand Index (KMeans k=3 vs GMM k=3): 0.2761
(ARI=1.0 = perfect agreement, 0.0 = random)


✓ fig_gmm_robustness.png


In [6]:
# ============================================================
# SECTION 4 — PREDICTIVE MODELLING (GroupKFold — Option Y)
# ============================================================

print('\n=== Section 4: GroupKFold CV ===')

df_rf = df[RF_FEATS + [OUTCOME, FARMER_ID]].apply(
    pd.to_numeric, errors='coerce').dropna().copy()
X_rf   = df_rf[RF_FEATS].values
y_rf   = df_rf[OUTCOME].values
groups = df_rf[FARMER_ID].values

print(f"RF dataset: {len(df_rf)} rows, {df_rf[FARMER_ID].nunique()} farmers")

gkfold = GroupKFold(n_splits=5)

print(f"\n{'Model':<28} {'R²':>8} {'±SD':>7} {'RMSE':>8} {'MAE':>8}")
print('─'*57)
model_results = {}
for name, model in [
    ('Linear Regression', LinearRegression()),
    ('Ridge Regression',  Ridge(alpha=1.0)),
    ('Random Forest (GroupKFold)',
     RandomForestRegressor(**RF_PARAMS)),
]:
    r2   = cross_val_score(model, X_rf, y_rf, cv=gkfold,
                           groups=groups, scoring='r2')
    rmse = -cross_val_score(model, X_rf, y_rf, cv=gkfold, groups=groups,
                            scoring='neg_root_mean_squared_error')
    mae  = -cross_val_score(model, X_rf, y_rf, cv=gkfold, groups=groups,
                            scoring='neg_mean_absolute_error')
    model_results[name] = dict(R2=r2.mean(), SD=r2.std(),
                               RMSE=rmse.mean(), MAE=mae.mean())
    marker = ' ←' if 'Forest' in name else ''
    print(f"{name:<28} {r2.mean():>8.3f} {r2.std():>7.3f} "
          f"{rmse.mean():>8.3f} {mae.mean():>8.3f}{marker}")

# Train final RF on full data for SHAP
rf_final = RandomForestRegressor(**RF_PARAMS)
rf_final.fit(df_rf[RF_FEATS], y_rf)


=== Section 4: GroupKFold CV ===
RF dataset: 832 rows, 417 farmers

Model                              R²     ±SD     RMSE      MAE
─────────────────────────────────────────────────────────
Linear Regression               0.257   0.028    0.599    0.468
Ridge Regression                0.257   0.028    0.599    0.468


Random Forest (GroupKFold)      0.283   0.036    0.588    0.455 ←


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsa

In [7]:
# ============================================================
# SECTION 4b — HYPERPARAMETER GRID SEARCH + ROBUSTNESS CHECK
# ============================================================

print('\n=== Section 4b: Grid search ===')

from sklearn.model_selection import GroupKFold
from itertools import product

param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [5, 8, None],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 0.5],
}

gkfold4b = GroupKFold(n_splits=5)
grid_results = []
for n_est, max_d, min_leaf, max_feat in product(
        param_grid['n_estimators'], param_grid['max_depth'],
        param_grid['min_samples_leaf'], param_grid['max_features']):
    rf_grid = RandomForestRegressor(
        n_estimators=n_est, max_depth=max_d,
        min_samples_leaf=min_leaf, max_features=max_feat,
        random_state=42, n_jobs=-1)
    r2_grid = cross_val_score(rf_grid, X_rf, y_rf, cv=gkfold4b,
                              groups=groups, scoring='r2')
    grid_results.append({
        'n_estimators': n_est, 'max_depth': max_d,
        'min_samples_leaf': min_leaf, 'max_features': max_feat,
        'R2_mean': r2_grid.mean(), 'R2_std': r2_grid.std()
    })

grid_df = pd.DataFrame(grid_results).sort_values('R2_mean', ascending=False)
print(f"Grid search: {len(grid_df)} combinations tested\n")
print(grid_df.head(10).to_string(index=False))

# Selected (reported) settings: n_estimators=500, min_samples_leaf=10,
# max_features='sqrt', max_depth=None — see Table: Random Forest
# hyperparameter optimization in the manuscript.
selected = grid_df[
    (grid_df.n_estimators == 500) &
    (grid_df.min_samples_leaf == 10) &
    (grid_df.max_features == 'sqrt') &
    (grid_df.max_depth.isna())
]
print(f"\nReported settings (n_estimators=500, min_samples_leaf=10, "
      f"max_features='sqrt', max_depth=None):")
print(selected.to_string(index=False))

# ── Fixed-hyperparameter robustness check ────────────────────────────────
# Verifies that the reported R^2 is not an artefact of hyperparameter
# search noise: refit the exact selected configuration once more and
# report its mean +/- SD, matching the figure quoted in the manuscript
# (R^2 = 0.283 +/- 0.036).
print('\n--- Fixed-hyperparameter robustness check ---')
rf_robust = RandomForestRegressor(**RF_PARAMS)
r2_robust = cross_val_score(rf_robust, X_rf, y_rf, cv=gkfold4b,
                            groups=groups, scoring='r2')
print(f"Fixed RF_PARAMS -> R2 = {r2_robust.mean():.3f} +/- {r2_robust.std():.3f}")
print("(Matches Table: RF (ours) R2 = 0.283 (0.036) reported in the manuscript)")



=== Section 4b: Grid search ===


Grid search: 36 combinations tested

 n_estimators  max_depth  min_samples_leaf max_features  R2_mean   R2_std
          500        8.0                 5          0.5 0.299517 0.035857
          300        8.0                 5          0.5 0.299371 0.035911
          500        NaN                 5          0.5 0.298135 0.039052
          300        NaN                 5          0.5 0.296916 0.040440
          300        NaN                10          0.5 0.295281 0.040769
          500        NaN                10          0.5 0.295012 0.041007
          300        8.0                10          0.5 0.294542 0.038337
          500        8.0                10          0.5 0.294497 0.038779
          300        5.0                 5          0.5 0.292654 0.036979
          500        5.0                 5          0.5 0.292257 0.036485

Reported settings (n_estimators=500, min_samples_leaf=10, max_features='sqrt', max_depth=None):
 n_estimators  max_depth  min_samples_leaf max_featu

Fixed RF_PARAMS -> R2 = 0.283 +/- 0.036
(Matches Table: RF (ours) R2 = 0.283 (0.036) reported in the manuscript)


In [8]:
# ============================================================
# SECTION 5 — SHAP ANALYSIS
# ============================================================

print('\n=== Section 5: SHAP ===')

df_shap = df[SHAP_FEATS + [OUTCOME, FARMER_ID]].apply(
    pd.to_numeric, errors='coerce').dropna().copy()
X_shap  = df_shap[SHAP_FEATS].rename(columns=SHAP_RENAME)
y_shap  = df_shap[OUTCOME].values

rf_shap = RandomForestRegressor(**RF_PARAMS)
rf_shap.fit(X_shap, y_shap)

explainer = shap.TreeExplainer(rf_shap)
sv        = explainer.shap_values(X_shap)
sv_rf     = sv        # alias for robustness check
exp_rf    = explainer

mean_shap = pd.Series(np.abs(sv).mean(0), index=X_shap.columns)

# Global bar
mean_shap_sorted = mean_shap.sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(mean_shap_sorted.index, mean_shap_sorted.values,
        color='#2980b9', edgecolor='white', linewidth=0.5)
ax.set_xlabel('mean(|SHAP value|)', fontsize=10)
ax.set_title('SHAP Feature Importance → Ch_Skill', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig3_importance_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig3_importance_new.png')

# Beeswarm
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_shap, plot_type='dot',
                  max_display=10, show=False)
plt.title('SHAP Beeswarm: Determinants of Skill Change', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4_shap_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig4_shap_new.png')

# Attach cluster labels
df_shap['Cluster'] = df_full.loc[
    df_shap.index.intersection(df_full.index), 'Cluster']
df_shap = df_shap.dropna(subset=['Cluster'])
df_shap['Cluster'] = df_shap['Cluster'].astype(int)

# Cluster-level beeswarm
panel_titles = [
    f'Cluster 1: Low-skill\n(n={(df_shap["Cluster"]==0).sum()})',
    f'Cluster 2: Moderate-skill\n(n={(df_shap["Cluster"]==1).sum()})',
    f'Cluster 3: High-skill\n(n={(df_shap["Cluster"]==2).sum()})',
]
fig = plt.figure(figsize=(17, 5))
for ci in range(3):
    ax  = fig.add_subplot(1, 3, ci+1)
    idx = df_shap[df_shap['Cluster']==ci].index
    sv_sub = explainer.shap_values(X_shap.loc[idx])
    top5   = (pd.Series(np.abs(sv_sub).mean(0), index=X_shap.columns)
                .sort_values(ascending=False).head(4).index.tolist())
    col_idx = [list(X_shap.columns).index(f) for f in top5]
    plt.sca(ax)
    shap.summary_plot(sv_sub[:, col_idx],
                      X_shap.loc[idx, top5].values,
                      feature_names=top5,
                      plot_type='dot', show=False,
                      plot_size=None, max_display=5)
    ax.set_title(panel_titles[ci], fontsize=11, fontweight='bold', pad=8)
    ax.tick_params(labelsize=9)
plt.suptitle('Cluster-Stratified SHAP: Segment-Specific Drivers',
             fontsize=12, y=1.03)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4b_cluster_shap_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig4b_cluster_shap_new.png')

# ── 5f. SHAP Robustness: RF vs XGBoost ───────────────────────────────────
print('\n--- 5f: SHAP Robustness ---')
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05,
                   max_depth=6, subsample=0.8, colsample_bytree=0.8,
                   random_state=42, n_jobs=-1, verbosity=0)
xgb.fit(X_shap, y_shap)
exp_xgb = shap.TreeExplainer(xgb)
sv_xgb  = exp_xgb.shap_values(X_shap)

imp_rf  = pd.Series(np.abs(sv_rf ).mean(0), index=X_shap.columns, name='RF')
imp_xgb = pd.Series(np.abs(sv_xgb).mean(0), index=X_shap.columns, name='XGBoost')
rho, pval = spearmanr(imp_rf.rank(ascending=False),
                      imp_xgb.rank(ascending=False))
top4_rf  = imp_rf.sort_values(ascending=False).head(4).index.tolist()
top4_xgb = imp_xgb.sort_values(ascending=False).head(4).index.tolist()
overlap  = set(top4_rf) & set(top4_xgb)
print(f"Spearman ρ = {rho:.4f}  (p={pval:.4e})")
print(f"Top-4 overlap: {sorted(overlap)}  ({len(overlap)}/4)")

compare = pd.concat([imp_rf, imp_xgb], axis=1).sort_values('RF', ascending=True)
fig, ax = plt.subplots(figsize=(9, 5.5))
y_pos   = np.arange(len(compare)); h = 0.36
ax.barh(y_pos+h/2, compare['RF'],      height=h,
        color='#1565C0', alpha=0.85, label='Random Forest')
ax.barh(y_pos-h/2, compare['XGBoost'], height=h,
        color='#00796B', alpha=0.85, label='XGBoost')
ax.set_yticks(y_pos); ax.set_yticklabels(compare.index, fontsize=10)
ax.set_xlabel('mean(|SHAP value|)', fontsize=10)
ax.set_title(f'SHAP Attribution Robustness: RF vs XGBoost\n'
             f'Spearman ρ = {rho:.3f}  |  Top-4 overlap = {len(overlap)}/4',
             fontsize=11)
ax.legend(fontsize=10, loc='lower right')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig_shap_robustness.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig_shap_robustness.png')


=== Section 5: SHAP ===


✓ fig3_importance_new.png


✓ fig4_shap_new.png


✓ fig4b_cluster_shap_new.png

--- 5f: SHAP Robustness ---


Spearman ρ = 0.7055  (p=4.8196e-03)
Top-4 overlap: ['All_Skill', 'Avg_ProdManage']  (2/4)


✓ fig_shap_robustness.png


In [9]:
# ── 5g. SHAP attribution mass for top-4 features (per cluster) ──────────────
print('\n--- 5g: SHAP Attribution Mass (Top-4 features per cluster) ---')

# คำนวณ attribution mass สำหรับแต่ละ cluster
attribution_mass = []
for ci in range(3):
    idx = df_shap[df_shap['Cluster'] == ci].index
    if len(idx) == 0:
        continue
    sv_sub = explainer.shap_values(X_shap.loc[idx])
    mean_abs_shap = np.abs(sv_sub).mean(0)
    total_mass = mean_abs_shap.sum()
    
    # top-4 features ตาม paper
    top4_cluster = ['All_Skill', 'Avg_ProdManage', 'Education', 'Avg_Mkting']
    # หา index ของ features เหล่านี้ใน X_shap.columns
    top4_idx = [list(X_shap.columns).index(f) for f in top4_cluster if f in X_shap.columns]
    top4_mass = mean_abs_shap[top4_idx].sum()
    pct = (top4_mass / total_mass) * 100
    
    cluster_name = CLUSTER_NAMES[ci]
    print(f'{cluster_name:>15}: top-4 features capture {pct:.1f}% of SHAP attribution mass')
    attribution_mass.append({'cluster': cluster_name, 'percentage': pct})

print(f'\nAverage across clusters: {np.mean([m["percentage"] for m in attribution_mass]):.1f}%')


--- 5g: SHAP Attribution Mass (Top-4 features per cluster) ---


      Low-skill: top-4 features capture 62.4% of SHAP attribution mass


 Moderate-skill: top-4 features capture 60.0% of SHAP attribution mass


     High-skill: top-4 features capture 60.6% of SHAP attribution mass

Average across clusters: 61.0%


In [10]:
# ============================================================
# SECTION 6 — SURROGATE DECISION TREE
# ============================================================

print('\n=== Section 6: Surrogate Decision Tree ===')

# SHAP-selected top-5 features
TOP4_FEATS = ['All_Skill', 'Avg_ProdManage', 'edu', 'Avg_Mkting']
top_feats = TOP4_FEATS  # override with fixed top-4 for consistency
# top_feats = mean_shap.sort_values(ascending=False).head(5).index.tolist()
feat_orig = [k for k, v in SHAP_RENAME.items() if v in top_feats]
print(f"Top-5 SHAP features: {top_feats}")

# ── 6a. In-sample (baseline — existing result) ────────────────────────────
X_tree_all = df_shap[feat_orig].rename(columns=SHAP_RENAME)
y_tree_all = (df_shap['Cluster'] == 2).astype(int)
farmer_all = df_shap[FARMER_ID].values

dt_full = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_full.fit(X_tree_all, y_tree_all)
acc_insample = accuracy_score(y_tree_all, dt_full.predict(X_tree_all))
print(f"\n6a. In-sample accuracy: {acc_insample:.3f}  (n={len(X_tree_all)})")

# ── Figure: Surrogate tree ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(dt_full, feature_names=top_feats,
          class_names=['Low/Moderate','High'],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title(f'Surrogate Decision Tree (depth=3, in-sample acc={acc_insample:.3f})',
             fontsize=12, pad=15)
plt.tight_layout()
plt.savefig(f'{OUT}/fig5_rules_new.png',
            bbox_inches='tight', dpi=180, facecolor='white')
plt.close()
print('✓ fig5_rules_new.png')

# ── 6b. Farmer-level 80/20 train/test split (Option B) ───────────────────
print('\n6b. Farmer-level 80/20 held-out validation (Option B)')

# Clustering features available in SHAP dataset
SURR_CLUSTER_FEATS = ['age','edu','agri_long','irriga','loan',
                      'Avg_ProdManage','Avg_Tech','Avg_Mkting',
                      'Ave_MktRisk','Ave_FinRisk']

# Year=1 only — one row per farmer (true post-training outcome)
ALL_FEATS_Y1 = SHAP_FEATS + [OUTCOME, FARMER_ID]
df_y0 = df[df['Year']==0][ALL_FEATS_Y1].apply(
    pd.to_numeric, errors='coerce').dropna(
    subset=SHAP_FEATS+[OUTCOME]).drop_duplicates(
    subset=FARMER_ID).reset_index(drop=True)

unique_farmers = df_y0[FARMER_ID].unique()
np.random.seed(42); np.random.shuffle(unique_farmers)
n_train    = int(len(unique_farmers) * 0.80)
train_ids  = unique_farmers[:n_train]
test_ids   = unique_farmers[n_train:]

df_tr = df_y0[df_y0[FARMER_ID].isin(train_ids)].copy()
df_te = df_y0[df_y0[FARMER_ID].isin(test_ids )].copy()
print(f"Train: {len(df_tr)} farmers | Test: {len(df_te)} farmers")

# Cluster on TRAIN only (Option B)
scaler_tr    = StandardScaler()
X_tr_sc      = scaler_tr.fit_transform(df_tr[SURR_CLUSTER_FEATS])
km_tr        = KMeans(n_clusters=3, random_state=42, n_init=10)
df_tr = df_tr.copy()
df_tr['Cluster_raw'] = km_tr.fit_predict(X_tr_sc)
tr_order   = df_tr.groupby('Cluster_raw')['All_Skill'].mean().sort_values()
tr_lbl_map = {old: new for new, old in enumerate(tr_order.index)}
df_tr['Cluster'] = df_tr['Cluster_raw'].map(tr_lbl_map)
print(f"Train cluster sizes: {df_tr['Cluster'].value_counts().sort_index().to_dict()}")

# Assign test to nearest centroid
df_te = df_te.copy()
df_te['Cluster_raw'] = km_tr.predict(
    scaler_tr.transform(df_te[SURR_CLUSTER_FEATS]))
df_te['Cluster'] = df_te['Cluster_raw'].map(tr_lbl_map)
print(f"Test  cluster sizes: {df_te['Cluster'].value_counts().sort_index().to_dict()}")

# SHAP on TRAIN to select top-4 features
X_tr_shap = df_tr[SHAP_FEATS].rename(columns=SHAP_RENAME)
rf_tr     = RandomForestRegressor(**RF_PARAMS)
rf_tr.fit(X_tr_shap, df_tr[OUTCOME].values)
sv_tr     = shap.TreeExplainer(rf_tr).shap_values(X_tr_shap)
top5_tr   = (pd.Series(np.abs(sv_tr).mean(0), index=X_tr_shap.columns)
               .sort_values(ascending=False).head(4).index.tolist())
feat_orig_tr = [k for k, v in SHAP_RENAME.items() if v in top5_tr]
print(f"Train top-5 SHAP: {top5_tr}")

# Surrogate tree — TRAIN
X_surr_tr = df_tr[feat_orig_tr].rename(columns=SHAP_RENAME)
y_surr_tr = (df_tr['Cluster'] == 2).astype(int)
dt_val    = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_val.fit(X_surr_tr, y_surr_tr)
acc_train = accuracy_score(y_surr_tr, dt_val.predict(X_surr_tr))

# Surrogate tree — TEST
X_surr_te = df_te[feat_orig_tr].rename(columns=SHAP_RENAME)
y_surr_te = (df_te['Cluster'] == 2).astype(int)
acc_test  = accuracy_score(y_surr_te, dt_val.predict(X_surr_te))

print(f"\nTrain accuracy (80%):    {acc_train:.3f}  (n={len(df_tr)})")
print(f"Held-out test (20%):     {acc_test:.3f}  (n={len(df_te)})")
print(f"\n{classification_report(y_surr_te, dt_val.predict(X_surr_te), target_names=['Low/Mod','High'])}")

# ── 6c. 5-fold GroupKFold CV ──────────────────────────────────────────────
print('\n6c. 5-fold GroupKFold CV on surrogate tree')

X_cv_all = df_shap[feat_orig_tr].rename(columns=SHAP_RENAME)
y_cv_all = (df_shap['Cluster'] == 2).astype(int)
g_cv_all = df_shap[FARMER_ID].values

gkf_surr = GroupKFold(n_splits=5)
cv_accs  = []
for fold, (tri, tei) in enumerate(gkf_surr.split(
        X_cv_all, y_cv_all, groups=g_cv_all)):
    dt_f = DecisionTreeClassifier(max_depth=3, random_state=42)
    dt_f.fit(X_cv_all.iloc[tri], y_cv_all.iloc[tri])
    a = accuracy_score(y_cv_all.iloc[tei],
                       dt_f.predict(X_cv_all.iloc[tei]))
    cv_accs.append(a)
    print(f"  Fold {fold+1}: {a:.3f}")
print(f"\n5-fold GroupKFold CV: {np.mean(cv_accs):.3f} +/- {np.std(cv_accs):.3f}")

print("\n=== FINAL SUMMARY: Surrogate Tree Accuracy ===")
print(f"{'In-sample (full, n=817)':<40} {acc_insample:.3f}")
print(f"{'Train set (80% farmers)':<40} {acc_train:.3f}")
print(f"{'Held-out test (20% farmers)':<40} {acc_test:.3f}")
print(f"{'5-fold GroupKFold CV':<40} {np.mean(cv_accs):.3f} +/- {np.std(cv_accs):.3f}")


=== Section 6: Surrogate Decision Tree ===
Top-5 SHAP features: ['All_Skill', 'Avg_ProdManage', 'edu', 'Avg_Mkting']

6a. In-sample accuracy: 0.933  (n=817)


✓ fig5_rules_new.png

6b. Farmer-level 80/20 held-out validation (Option B)
Train: 332 farmers | Test: 84 farmers
Train cluster sizes: {0: 98, 1: 100, 2: 134}
Test  cluster sizes: {0: 34, 1: 30, 2: 20}


Train top-5 SHAP: ['All_Skill', 'Avg_Mkting', 'Avg_ProdManage', 'Avg_Tech']

Train accuracy (80%):    0.919  (n=332)
Held-out test (20%):     0.845  (n=84)

              precision    recall  f1-score   support

     Low/Mod       0.93      0.86      0.89        64
        High       0.64      0.80      0.71        20

    accuracy                           0.85        84
   macro avg       0.79      0.83      0.80        84
weighted avg       0.86      0.85      0.85        84


6c. 5-fold GroupKFold CV on surrogate tree
  Fold 1: 0.927
  Fold 2: 0.921
  Fold 3: 0.933
  Fold 4: 0.914
  Fold 5: 0.890

5-fold GroupKFold CV: 0.917 +/- 0.015

=== FINAL SUMMARY: Surrogate Tree Accuracy ===
In-sample (full, n=817)                  0.933
Train set (80% farmers)                  0.919
Held-out test (20% farmers)              0.845
5-fold GroupKFold CV                     0.917 +/- 0.015


In [11]:
# ============================================================
# SECTION 7 — ABLATION STUDY
# ============================================================
# Compares the full SAR pipeline against two simplified baselines:
#   (1) Clustering-only  — threshold on All_Skill, no ML attribution
#   (2) RF-only           — global RF prediction threshold, no segmentation
#   (3) SAR (proposed)    — segmentation + SHAP + surrogate rules
# Reuses df_shap, X_shap, y_tree, feat_orig, dt_full, acc_insample
# already defined in Section 6.

print('\n=== Section 7: Ablation study ===')

# ── (1) Clustering-only: threshold on overall skill (All_Skill) ──────────
# Uses the mean All_Skill across the sample as a simple, parameter-free
# cut-off — no machine learning involved.
all_skill_vals = df_shap['All_Skill'].values
thresh_cluster = all_skill_vals.mean()
pred_cluster   = (all_skill_vals > thresh_cluster).astype(int)
acc_cluster    = accuracy_score(y_tree_all, pred_cluster)
print(f"[1] Clustering-only (All_Skill > {thresh_cluster:.3f}): "
      f"accuracy = {acc_cluster:.3f}")

# ── (2) RF-only: global prediction threshold, no segmentation ────────────
# Uses the Random Forest's continuous Ch_Skill prediction with a simple
# zero/median cut-off to flag 'high training potential' farmers, without
# any cluster-specific structure.
rf_pred_all     = rf_shap.predict(X_shap)
thresh_rf       = np.median(rf_pred_all)
pred_rf         = (rf_pred_all > thresh_rf).astype(int)
acc_rf_only     = accuracy_score(y_tree_all, pred_rf)
print(f"[2] RF-only (pred > {thresh_rf:.3f}): "
      f"accuracy = {acc_rf_only:.3f}")

# ── (3) SAR (proposed): segmentation + SHAP + surrogate rules ────────────
print(f"[3] SAR (proposed): accuracy = {acc_insample:.3f}")

# ── Summary table (matches Table: Ablation study in the manuscript) ─────
ablation_df = pd.DataFrame([
    {'Model': 'Clustering-only', 'Description': 'Baseline skill threshold',
     'Accuracy': acc_cluster},
    {'Model': 'RF (no seg.)',    'Description': 'Global prediction threshold',
     'Accuracy': acc_rf_only},
    {'Model': 'SAR (proposed)',  'Description': 'Segmentation + SHAP + rules',
     'Accuracy': acc_insample},
])
ablation_df['Accuracy (%)'] = (ablation_df['Accuracy'] * 100).round(1)
print("\n" + ablation_df[['Model', 'Description', 'Accuracy (%)']].to_string(index=False))



=== Section 7: Ablation study ===
[1] Clustering-only (All_Skill > 3.587): accuracy = 0.891
[2] RF-only (pred > 0.001): accuracy = 0.793
[3] SAR (proposed): accuracy = 0.933

          Model                 Description  Accuracy (%)
Clustering-only    Baseline skill threshold          89.1
   RF (no seg.) Global prediction threshold          79.3
 SAR (proposed) Segmentation + SHAP + rules          93.3


In [12]:
# ============================================================
# SUMMARY
# ============================================================
print('\n' + '='*55)
print('All figures saved to ./figures/')
print('='*55)
for fn, desc in [
    ('fig_elbow_silhouette.png',   '4-panel cluster validity'),
    ('fig2_clusters_new.png',      'PCA scatter k=3'),
    ('fig_k_sensitivity.png',      'k-sensitivity k=2..5  [NEW]'),
    ('fig_gmm_robustness.png',     'GMM vs K-Means  [NEW]'),
    ('fig3_importance_new.png',    'Global SHAP bar'),
    ('fig4_shap_new.png',          'Global SHAP beeswarm'),
    ('fig4b_cluster_shap_new.png', 'Cluster SHAP 3-panel'),
    ('fig_shap_robustness.png',    'RF vs XGBoost SHAP'),
    ('fig5_rules_new.png',         'Surrogate decision tree'),
]:
    print(f"  {fn:<38} {desc}")


All figures saved to ./figures/
  fig_elbow_silhouette.png               4-panel cluster validity
  fig2_clusters_new.png                  PCA scatter k=3
  fig_k_sensitivity.png                  k-sensitivity k=2..5  [NEW]
  fig_gmm_robustness.png                 GMM vs K-Means  [NEW]
  fig3_importance_new.png                Global SHAP bar
  fig4_shap_new.png                      Global SHAP beeswarm
  fig4b_cluster_shap_new.png             Cluster SHAP 3-panel
  fig_shap_robustness.png                RF vs XGBoost SHAP
  fig5_rules_new.png                     Surrogate decision tree
